In [1]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import TemporalFusionTransformer
from torch.utils.data import DataLoader

print("PyTorch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---- PATHS ----
DATA_H1 = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet"
DATA_H4 = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H4.parquet"

CKPT_MT = r"C:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_73\checkpoints\eurusd_mt_h1_to_h4-epoch=28-val_loss=0.001256.ckpt"
assert os.path.exists(DATA_H1)
assert os.path.exists(DATA_H4)
assert os.path.exists(CKPT_MT)
print("✅ All files found.")


PyTorch: 2.5.1+cu121
Using device: cuda
✅ All files found.


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:28: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [4]:
import numpy as np
import pandas as pd

# --- Paths (you already defined DATA_H1, DATA_H4 above) ---
print("Loading H1 from:", DATA_H1)
print("Loading H4 from:", DATA_H4)

df_h1 = pd.read_parquet(DATA_H1)
df_h4 = pd.read_parquet(DATA_H4)

print("H1 columns:", df_h1.columns.tolist())
print("H4 columns:", df_h4.columns.tolist())
print("H1 shape:", df_h1.shape)
print("H4 shape:", df_h4.shape)

# --- Ensure datetime ---
df_h1["time"] = pd.to_datetime(df_h1["time"])
df_h4["time"] = pd.to_datetime(df_h4["time"])

# --- Ensure OHLC in H1 (for log_return_h1) ---
required_ohlc = ["mid_o", "mid_h", "mid_l", "mid_c"]
for c in required_ohlc:
    if c not in df_h1.columns:
        raise ValueError(f"H1 OHLC missing column: {c}")

# --- Create log_return_h1 if missing ---
if "log_return_h1" not in df_h1.columns:
    print("⚠️ log_return_h1 not found in H1 — creating it now.")
    df_h1 = df_h1.sort_values("time").reset_index(drop=True)
    df_h1["close_shift_h1"] = df_h1["mid_c"].shift(1)
    df_h1["log_return_h1"] = np.log(df_h1["mid_c"] / df_h1["close_shift_h1"])
    df_h1.drop(columns=["close_shift_h1"], inplace=True)
else:
    print("✅ log_return_h1 already present in H1 data.")

# --- Ensure OHLC in H4 (for log_return_4h) ---
for c in required_ohlc:
    if c not in df_h4.columns:
        raise ValueError(f"H4 OHLC missing column: {c}")

# --- Create log_return_4h if missing ---
if "log_return_4h" not in df_h4.columns:
    print("⚠️ log_return_4h not found in H4 — creating it now.")
    df_h4 = df_h4.sort_values("time").reset_index(drop=True)
    df_h4["close_shift_h4"] = df_h4["mid_c"].shift(1)
    df_h4["log_return_4h"] = np.log(df_h4["mid_c"] / df_h4["close_shift_h4"])
    df_h4.drop(columns=["close_shift_h4"], inplace=True)
else:
    print("✅ log_return_4h already present in H4 data.")

# --- Merge H1 + H4 on time ---
df = df_h1.merge(
    df_h4[["time", "log_return_4h"]],
    on="time",
    how="inner",
)

df = df.sort_values("time").reset_index(drop=True)

# --- Add time_idx + series_id ---
df["time_idx"] = np.arange(len(df), dtype="int64")
df["series_id"] = 0

print("Merged df shape:", df.shape)
df.head()


Loading H1 from: C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet
Loading H4 from: C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H4.parquet
H1 columns: ['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c']
H4 columns: ['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c']
H1 shape: (61473, 14)
H4 shape: (15375, 14)
⚠️ log_return_h1 not found in H1 — creating it now.
⚠️ log_return_4h not found in H4 — creating it now.
Merged df shape: (15365, 18)


,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c,log_return_h1,log_return_4h,time_idx,series_id
0,2016-01-07 02:00:00+00:00,1567,1.08026,1.08176,1.07996,1.08152,1.08018,1.08169,1.07987,1.08144,1.08035,1.08184,1.08005,1.08159,0.001138,0.002247,0,0
1,2016-01-07 06:00:00+00:00,788,1.08274,1.08280,1.08118,1.08170,1.08267,1.08272,1.08110,1.08159,1.08281,1.08287,1.08125,1.08180,-0.000943,0.001735,1,0
2,2016-01-07 10:00:00+00:00,2092,1.08458,1.08596,1.08350,1.08374,1.08447,1.08589,1.08342,1.08367,1.08469,1.08604,1.08358,1.08381,-0.000793,0.001179,2,0
3,2016-01-07 14:00:00+00:00,2917,1.08584,1.08645,1.08366,1.08441,1.08577,1.08638,1.08360,1.08434,1.08591,1.08653,1.08373,1.08448,-0.001355,0.000350,3,0
4,2016-01-07 18:00:00+00:00,2364,1.08628,1.09092,1.08622,1.09018,1.08621,1.09083,1.08615,1.09009,1.08636,1.09103,1.08630,1.09026,0.003602,0.006259,4,0


In [5]:
max_encoder_length = 96    # H1 history window
max_prediction_length = 1  # predict next 4H move

mt_dataset = TimeSeriesDataSet(
    df,
    time_idx="time_idx",
    target="log_return_4h",      # 4H log-return as target
    group_ids=["series_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    # unknown reals (encoder + decoder)
    time_varying_unknown_reals=[
        "log_return_h1",
        "mid_o", "mid_h", "mid_l", "mid_c",
    ],
    # known reals (we only used time_idx as simple positional feature)
    time_varying_known_reals=["time_idx"],
)

# Train/val split (same logic you used: 80/20 split on time_idx)
cutoff = int(len(df) * 0.8)
training = mt_dataset.filter(lambda x: x["time_idx"] <= cutoff)
validation = mt_dataset.filter(lambda x: x["time_idx"] > cutoff)

train_dataloader = DataLoader(training, batch_size=64, shuffle=False, num_workers=0)
val_dataloader   = DataLoader(validation, batch_size=64, shuffle=False, num_workers=0)

print("Training samples:", len(training))
print("Validation samples:", len(validation))

example = training[0]
print("Example keys:", example[0].keys())


KeyError: 'time_idx'

In [ ]:
tft_mt = TemporalFusionTransformer.load_from_checkpoint(
    CKPT_MT,
    dataset=training,      # very important so encoders & scalers match
    map_location=device,
)

tft_mt.to(device)
tft_mt.eval()

print("✅ Model loaded on", device)
print("Model parameters (k):", tft_mt.size() / 1e3)


In [ ]:
@torch.no_grad()
def get_preds_vs_true_from_loader(model, dataloader, device):
    model.eval()
    all_time_idx = []
    all_true = []
    all_pred = []

    for batch in dataloader:
        x, y = batch  # y is usually (target, weight)

        # Extract target tensor
        if isinstance(y, (list, tuple)):
            y = y[0]
        if not isinstance(y, torch.Tensor):
            raise RuntimeError(f"Unexpected y type: {type(y)}")

        # Move to device
        x = {
            k: (v.to(device) if isinstance(v, torch.Tensor) else v)
            for k, v in x.items()
        }
        y = y.to(device)

        # Forward pass
        out = model(x)
        # out["prediction"]: [batch, max_pred_len, output_size]
        preds = out["prediction"]  # shape [B, 1, 1] in your config

        # Take last decoder step and first output dimension
        preds_last = preds[:, -1, 0]   # [B]
        true_last  = y[:, -1]          # [B]

        # time index for the decoded step
        time_idx_dec = x["decoder_time_idx"][:, -1]  # [B]

        # Move to CPU numpy
        all_time_idx.append(time_idx_dec.detach().cpu().numpy())
        all_true.append(true_last.detach().cpu().numpy())
        all_pred.append(preds_last.detach().cpu().numpy())

    time_idx = np.concatenate(all_time_idx)
    y_true   = np.concatenate(all_true)
    y_pred   = np.concatenate(all_pred)

    # Sort by time_idx
    order = np.argsort(time_idx)
    return time_idx[order], y_true[order], y_pred[order]


time_idx_val, y_true_val, y_pred_val = get_preds_vs_true_from_loader(tft_mt, val_dataloader, device)

print("Validation preds shape:", y_pred_val.shape)
print("Validation true shape:", y_true_val.shape)
print("Time_idx shape:", time_idx_val.shape)


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

mse = mean_squared_error(y_true_val, y_pred_val)
mae = mean_absolute_error(y_true_val, y_pred_val)

# Direction accuracy (ignore exact zeros)
mask = y_true_val != 0
dir_acc = np.mean(np.sign(y_true_val[mask]) == np.sign(y_pred_val[mask])) * 100.0

# Approx pips (simple scale: 1 pip ≈ 1e-4)
approx_mae_pips = mae * 1e4

print("=== H1→H4 Multi-Timeframe TFT — Validation Metrics ===")
print(f"MSE: {mse:.8f}")
print(f"MAE: {mae:.8f}")
print(f"Direction accuracy (non-zero true): {dir_acc:.2f}%")
print(f"Approx MAE in pips (4H): {approx_mae_pips:.2f} pips")


In [ ]:
# Simple sign-based strategy:
# if prediction > 0  → go long
# if prediction < 0  → go short
# realized pnl (in pips) ≈ sign(pred) * true * 1e4

signal = np.sign(y_pred_val)
realized_pips = signal * y_true_val * 1e4  # pips per 4H bar

equity_curve = np.cumsum(realized_pips)

total_pips = equity_curve[-1]
mean_pips = realized_pips.mean()
std_pips = realized_pips.std()
sharpe_like = mean_pips / std_pips if std_pips > 0 else 0.0

print("=== H1→H4 Multi-Timeframe TFT — Simple Pips Backtest (Validation) ===")
print(f"Total pips: {total_pips:.2f}")
print(f"Average pips per trade: {mean_pips:.4f}")
print(f"Std pips per trade: {std_pips:.4f}")
print(f"Sharpe-like (mean/std): {sharpe_like:.4f}")


In [ ]:
N = 500  # last N points to visualize
idx_slice = -N if N < len(y_true_val) else 0

t_idx_plot = time_idx_val[idx_slice:]
true_plot = y_true_val[idx_slice:] * 1e4   # pips
pred_plot = y_pred_val[idx_slice:] * 1e4   # pips

plt.figure(figsize=(14, 5))
plt.plot(t_idx_plot, true_plot, label="True 4H return (pips)", alpha=0.7)
plt.plot(t_idx_plot, pred_plot, label="Predicted 4H return (pips)", alpha=0.7)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("H1→H4 TFT – Last {} validation points (pips)".format(len(t_idx_plot)))
plt.xlabel("time_idx")
plt.ylabel("Return (pips)")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(14, 5))
plt.plot(time_idx_val, equity_curve, label="Equity curve (pips)")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Simple sign-based strategy — validation equity curve")
plt.xlabel("time_idx")
plt.ylabel("Cumulative pips")
plt.legend()
plt.grid(True)
plt.show()
